# LatentR3-SFT + SigLIP en Toys_and_Games (extensión multimodal)

LatentR3 con una imagen por ítem del historial. Después de cada título va el marcador `<|image_pad|>`, un token que ya existe en el vocabulario de Qwen2.5, y su embedding se reemplaza por `proyector(SigLIP(imagen))`. El proyector es lineal (1152 → 1536). Los ítems sin imagen (9,5 % de Toys) usan un vector aprendido `sin_imagen`, compartido. El reemplazo se hace en las dos pasadas del modelo, así que el token latente también ve las imágenes. Todo lo demás es el SFT y la evaluación de `../latentr3/latentr3.ipynb`.

Es una **extensión propia**, no es parte del paper, y se reporta aparte de la replicación. El código está en `latentr3_mm.py`, que reutiliza `../latentr3/latentr3.py`. Las decisiones, fuentes y verificaciones están en `README.md`, en la sección "LatentR3-SFT + SigLIP".

**Escribe archivos** en `checkpoints/latentr3_mm/Toys_and_Games/`, que git ignora, y puede retomarse si se corta. Necesita `checkpoints/multimodal/Toys_and_Games/siglip.npz` (sha256 verificado).

## Entorno y configuración

In [ ]:
import os
os.environ['HF_HUB_OFFLINE'] = '1'   # Qwen2.5-1.5B ya está en la caché local
# Solo la GPU dedicada (índice 0), antes de importar torch; no combinar con HIP_VISIBLE_DEVICES (README de la raíz)
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')

import sys
import json
import shutil
import pandas as pd
import torch
import transformers
import trl

In [ ]:
base = os.path.join('..', '..', 'repo_latentr3', 'data')
latentr3 = os.path.abspath(os.path.join('..', '..', 'repo_latentr3'))
calc_py = os.path.join(latentr3, 'src', 'utils', 'calc.py')
# latentr3_mm.py importa latentr3.py (../latentr3), bigrec_baseline.py (../bigrec) y baseline_comun.py (..)
for carpeta in ['..', os.path.join('..', 'bigrec'), os.path.join('..', 'latentr3')]:
    carpeta = os.path.abspath(carpeta)
    if carpeta not in sys.path:
        sys.path.insert(0, carpeta)

import latentr3 as lr3
import latentr3_mm as mm
lr3.cargar_autores(latentr3)   # verifica los sha256 del código de los autores antes de importarlo
mm.cargar_features()           # verifica el sha256 de siglip.npz
CONFIG_SFT, CONFIG_RL = lr3.CONFIG_SFT, lr3.CONFIG_RL
r = mm.rutas()                 # checkpoints/latentr3_mm/Toys_and_Games/{sft,rl_1e-5}/...
print('Salida:', mm.SALIDA)
print('torch', torch.__version__, '| transformers', transformers.__version__, '| trl', trl.__version__)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), '| bf16:', torch.cuda.is_bf16_supported())
else:
    print('Sin GPU: solo se pueden correr las verificaciones sin GPU.')

## Verificaciones sin GPU

Solo CPU, con modelos Qwen diminutos de 2 capas; tardan segundos.
- **VM1:** `<|image_pad|>` es un solo token (151655), `<|Thought|>` sigue siendo el último del vocabulario y ningún título del catálogo contiene un token especial.
- **VM2:** en 200 filas por split, el prompt sin los marcadores es el de `LatentRDataset` (SFT) y el de `D3Dataset` (RL). Además, la cantidad de marcadores es igual a la de ítems del historial.
- **VM3:** si cada imagen se reemplaza por el embedding del propio marcador, los logits son iguales a los de `LatentModel`. El gradiente llega al proyector y a `sin_imagen`.
- **VM4:** `generate` entrega los `item_ids` una sola vez, al procesar el prompt, replicados por beam. Guardar y cargar el modelo no cambia los logits.
- **VM6:** el trainer del RL pasa los `item_ids` de cada grupo al modelo, y un paso solo cambia LatentRATT.

In [ ]:
tokenizer = lr3.tokenizer_con_latente()
print('VM1', mm.verificar_token(base, tokenizer))
display(pd.DataFrame(mm.verificar_dataset(base, tokenizer)).T)
print('VM3', mm.verificar_modelo())
print('VM4', mm.verificar_generacion(os.path.join(mm.SALIDA, '_vm4')))
print('VM6', mm.verificar_rl(base, lr3.tokenizer_con_latente(), os.path.join(mm.SALIDA, '_vm6')))

## Prueba rápida (VM5)

SFT con 1024 filas de `train` y 256 de `valid`, y evaluación de 100 filas de `test`, normal y sin imágenes. Además, RL con 64 prompts (2 pasos) y su evaluación. Comprueba que `item_ids` llega como tensor al trainer, que hay 0 títulos fuera del catálogo, que las métricas coinciden con `calc.py` y que el RL solo cambia LatentRATT. También mide tiempo y memoria. Escribe en `checkpoints/latentr3_mm/Toys_and_Games/_prueba/`, que se borra al final (`BORRAR_PRUEBA`). Duración estimada: 15–20 min de GPU (sin medir).

In [ ]:
PRUEBA = os.path.join(mm.SALIDA, '_prueba')
rp = mm.rutas(PRUEBA)
info = lr3.buscar_archivo(base, 'info', mm.CATEGORIA)
eval_prueba = {}
def evaluar_prueba(nombre, dir_modelo, dir_pred, lote, lp, sin_imagenes=False):
    ruta = mm.evaluar(base, dir_modelo, dir_pred, lote, lp, sin_imagenes=sin_imagenes, sample=100)
    with open(os.path.join(dir_pred, 'registro.json')) as f:
        registro = json.load(f)
    lr3.verificar_con_calc(ruta, info, calc_py)   # falla si las métricas no coinciden con calc.py
    eval_prueba[nombre] = {'s por fila': registro['segundos'] / registro['filas'],
                           'memoria máx. GB': registro.get('memoria_max_gb'), **lr3.metricas(ruta, info)}

registro_sft = mm.entrenar_sft(base, rp['sft'], sample_train=1024, sample_valid=256, **CONFIG_SFT)
display(pd.Series({k: registro_sft[k] for k in ['s_por_paso_mediana', 's_valid_por_fila', 'memoria_max_gb',
                                                 'memoria_reservada_max_gb', 'micro_batch', 'acumulacion']}))
evaluar_prueba('SFT + SigLIP', rp['modelo_sft'], rp['pred_sft'], lr3.BATCH_GEN_SFT, lr3.LENGTH_PENALTY['latentr3'])
evaluar_prueba('SFT + SigLIP, sin imágenes', rp['modelo_sft'], rp['pred_sin_imagenes'], lr3.BATCH_GEN_SFT,
               lr3.LENGTH_PENALTY['latentr3'], sin_imagenes=True)
registro_rl = mm.entrenar_rl(base, rp['rl'], rp['modelo_sft'], sample=64, **CONFIG_RL)
print('Solo cambió LatentRATT:', lr3.verificar_rl_congelado(rp['modelo_sft'], rp['modelo_rl']))
display(pd.Series({k: registro_rl[k] for k in ['pasos_totales', 's_por_paso_mediana', 'memoria_max_gb',
                                                'parametros_entrenables', 'recompensa_por_paso']}))
evaluar_prueba('RL', rp['modelo_rl'], rp['pred_rl'], lr3.BATCH_GEN_RL, lr3.LENGTH_PENALTY['latentr3'])
assert all(v['fuera del catálogo'] == 0 for v in eval_prueba.values())
BORRAR_PRUEBA = True
if BORRAR_PRUEBA:
    shutil.rmtree(PRUEBA, ignore_errors=True)
pd.DataFrame(eval_prueba).T

## Entrenamiento (SFT)

Es el SFT de LatentR3 con los mismos argumentos que en `../latentr3/latentr3.ipynb` (celda 18): lr 3e-4, batch 128, hasta 3 épocas, paciencia 1, AdamW con los estados en la RAM (F3), bf16. El proyector y `sin_imagen` se entrenan junto con todo el modelo. Escribe `sft/modelo/` (la mejor época, con los buffers SigLIP) y `sft/entrenamiento.json`. Si se corta, volver a correr la celda continúa desde el último checkpoint.

Duración estimada: ~7,2 h, sin medir. Son las 6,73 h del SFT de texto escaladas por tokens por fila: 167,33 frente a 155,92, porque cada imagen agrega ~1,5 tokens. GPU ~10 GB; RAM ~11 GB.

In [ ]:
registro_sft = mm.entrenar_sft(base, r['sft'], **CONFIG_SFT)
pd.Series({'eval_loss por época': [round(e['eval_loss'], 4) for e in registro_sft['evaluaciones']],
           'mejor eval_loss': registro_sft['mejor_eval_loss'], 'pasos': registro_sft['pasos_totales'],
           'h (última sesión)': registro_sft['segundos_esta_sesion'] / 3600,
           'memoria máx. GB': registro_sft['memoria_max_gb']})

## Evaluación

Igual que el "only SFT" de texto: 10 beams restringidos por el trie, `length_penalty` 1,0 y lotes de 12. La segunda celda es la **ablación**: el mismo modelo con todas las imágenes reemplazadas por el vector `sin_imagen`. Si las métricas no cambian, el modelo ignora las imágenes y la diferencia con el texto vendría de los tokens extra del prompt. Cada evaluación tarda ~40 min (estimado) y se retoma por partes.

In [ ]:
mm.evaluar(base, r['modelo_sft'], r['pred_sft'], lr3.BATCH_GEN_SFT, lr3.LENGTH_PENALTY['latentr3'])

In [ ]:
mm.evaluar(base, r['modelo_sft'], r['pred_sin_imagenes'], lr3.BATCH_GEN_SFT, lr3.LENGTH_PENALTY['latentr3'],
           sin_imagenes=True)

## Métricas y comparación

Cada `final_result.json` se verifica con `gao` de `calc.py` y debe tener 0 títulos fuera del catálogo. Se compara con el "only SFT" de texto y con nuestro BIGRec y D3 de Toys, todos con las predicciones versionadas en `resultados/`:
- comparación pareada fila a fila, con McNemar exacto y bootstrap de NDCG@10 (`../latentr3/analisis_pareado.py`);
- por popularidad del objetivo (≤ 5 interacciones en `train`, la definición de `sasrec_mm.py`);
- según si todo el historial tiene imagen.

Escribe `resultados/multimodal/Toys_and_Games/latentr3_mm/analisis.json` y copia allí `entrenamiento.json` y las predicciones (`final_result.json` y `registro.json`).

In [ ]:
RES = os.path.abspath(os.path.join('..', '..', 'resultados'))
predicciones = {
    'only SFT (texto)': os.path.join(RES, 'latentr3', mm.CATEGORIA, 'sft', 'predicciones', 'test', 'final_result.json'),
    'BIGRec': os.path.join(RES, 'bigrec', mm.CATEGORIA, 'predicciones', 'bigrec', 'final_result.json'),
    'D3': os.path.join(RES, 'bigrec', mm.CATEGORIA, 'predicciones', 'd3', 'final_result.json'),
    'SFT + SigLIP': os.path.join(r['pred_sft'], 'final_result.json'),
    'SFT + SigLIP, sin imágenes': os.path.join(r['pred_sin_imagenes'], 'final_result.json'),
}
for nombre in ['SFT + SigLIP', 'SFT + SigLIP, sin imágenes']:
    lr3.verificar_con_calc(predicciones[nombre], info, calc_py)
pares = [('only SFT (texto)', 'SFT + SigLIP'), ('SFT + SigLIP, sin imágenes', 'SFT + SigLIP'),
         ('only SFT (texto)', 'SFT + SigLIP, sin imágenes'), ('BIGRec', 'SFT + SigLIP')]
analisis = mm.analizar(base, predicciones, pares)
assert all(m['fuera del catálogo'] == 0 for m in analisis['métricas'].values())

# Copias versionadas
os.makedirs(mm.RESULTADOS, exist_ok=True)
shutil.copy2(os.path.join(r['sft'], 'entrenamiento.json'), os.path.join(mm.RESULTADOS, 'entrenamiento.json'))
for carpeta in ['test', 'test_sin_imagenes']:
    destino = os.path.join(mm.RESULTADOS, 'predicciones', carpeta)
    os.makedirs(destino, exist_ok=True)
    for archivo in ['final_result.json', 'registro.json']:
        shutil.copy2(os.path.join(r['sft'], 'predicciones', carpeta, archivo), os.path.join(destino, archivo))
with open(os.path.join(mm.RESULTADOS, 'analisis.json'), 'w') as f:
    json.dump(analisis, f, indent=1, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, 'item') else str(o))

metricas = ['HR@5', 'HR@10', 'NDCG@5', 'NDCG@10', 'fuera del catálogo', 'filas']
display(pd.DataFrame(analisis['métricas']).T[metricas].round(4))
display(pd.DataFrame({par: {'gana / pierde HR@10': f"{v['@10']['filas que gana']} / {v['@10']['filas que pierde']}",
                            'McNemar p HR@10': v['@10']['McNemar exacto p'],
                            'dif. NDCG@10': v['@10']['dif. NDCG'],
                            'IC95 dif. NDCG@10': v['@10']['dif. NDCG IC95 bootstrap']}
                      for par, v in analisis['pareado'].items()}).T)
pd.DataFrame(analisis['grupos']).T.filter(regex='filas|HR@10').round(4)

## RL multimodal (opcional)

El RL de LatentR3 (LR-GRPO, solo LatentRATT) se hace desde el SFT multimodal, con la configuración del RL de texto: lr 1e-5, 1 época y 2 GPUs simuladas. El proyector y `sin_imagen` quedan congelados. Después se evalúa con `length_penalty` 1,0 (LatentR3) y 0,0 (D3 + LatentR3).

**Se corre solo si queda tiempo:** cambiar `CORRER_RL = True`. Duración estimada: ~7,2 h de RL más ~1,4 h de evaluación, sin medir. En Toys, el RL de texto con 1e-5 no cambió las métricas, y con lr mayor colapsó (`../latentr3/README.md`, "Diagnóstico del RL").

In [ ]:
CORRER_RL = False
if CORRER_RL:
    registro_rl = mm.entrenar_rl(base, r['rl'], r['modelo_sft'], **CONFIG_RL)
    print('Solo cambió LatentRATT:', lr3.verificar_rl_congelado(r['modelo_sft'], r['modelo_rl']))
    mm.evaluar(base, r['modelo_rl'], r['pred_rl'], lr3.BATCH_GEN_RL, lr3.LENGTH_PENALTY['latentr3'])
    mm.evaluar(base, r['modelo_rl'], r['pred_d3'], lr3.BATCH_GEN_RL, lr3.LENGTH_PENALTY['d3'])
    resultados_rl = {}
    for nombre, clave in [('SFT + SigLIP', 'pred_sft'), ('RL + SigLIP', 'pred_rl'), ('D3 + RL + SigLIP', 'pred_d3')]:
        ruta = os.path.join(r[clave], 'final_result.json')
        lr3.verificar_con_calc(ruta, info, calc_py)
        resultados_rl[nombre] = lr3.metricas(ruta, info)
    display(pd.DataFrame(resultados_rl).T[['HR@5', 'HR@10', 'NDCG@5', 'NDCG@10', 'fuera del catálogo']].round(4))